In [0]:
train_df = spark.read.option("header", True).option("inferSchema", True).csv(
    "/Volumes/workspace/default/nyc_data_vol/silver_cmapss_train.csv"
)

test_df = spark.read.option("header", True).option("inferSchema", True).csv(
    "/Volumes/workspace/default/nyc_data_vol/silver_cmapss_test.csv"
)

rul_df = spark.read.option("header", True).option("inferSchema", True).csv(
    "/Volumes/workspace/default/nyc_data_vol/silver_cmapss_rul.csv"
)

In [0]:
display(train_df.limit(5))
display(test_df.limit(5))
display(rul_df.limit(5))

In [0]:
from pyspark.sql import functions as F

# Load the uploaded Silver-layer CSV files
train_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/default/nyc_data_vol/silver_cmapss_train.csv")
)

test_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/default/nyc_data_vol/silver_cmapss_test.csv")
)

rul_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/default/nyc_data_vol/silver_cmapss_rul.csv")
)

print("Train rows:", train_df.count())
print("Test rows:", test_df.count())
print("RUL rows:", rul_df.count())

train_df.printSchema()

Basic analytics

In [0]:
train_df.select("RUL").summary().show()

In [0]:
display(
    train_df.select("RUL")
)

Engine lifecycle analytics

In [0]:
engine_life = (
    train_df
    .groupBy("unit_number")
    .agg(
        F.max("cycle").alias("lifetime_cycles")
    )
    .orderBy("unit_number")
)

display(engine_life)

In [0]:
engine_life.select("lifetime_cycles").summary().show()

Analyze RUL over an engine's lifetime

In [0]:
selected_engines = [1, 10, 20, 50]

rul_progression = (
    train_df
    .filter(F.col("unit_number").isin(selected_engines))
    .select("unit_number", "cycle", "RUL")
    .orderBy("unit_number", "cycle")
)

display(rul_progression)

Find useful sensors

In [0]:
sensor_columns = [
    c for c in train_df.columns
    if c.startswith("sensor_")
]

correlations = []

for sensor in sensor_columns:
    corr_value = train_df.stat.corr(sensor, "RUL")
    correlations.append((sensor, corr_value))

correlation_df = spark.createDataFrame(
    correlations,
    ["sensor", "correlation_with_RUL"]
)

display(
    correlation_df
    .orderBy(F.abs(F.col("correlation_with_RUL")).desc())
)

In [0]:
sensor_stats = []

for sensor in sensor_columns:
    row = (
        train_df
        .select(
            F.mean(sensor).alias("mean"),
            F.stddev(sensor).alias("stddev")
        )
        .collect()[0]
    )

    sensor_stats.append(
        (sensor, row["mean"], row["stddev"])
    )

sensor_stats_df = spark.createDataFrame(
    sensor_stats,
    ["sensor", "mean", "stddev"]
)

display(sensor_stats_df.orderBy("stddev"))

Prepare ML features

In [0]:
feature_columns = [
    "cycle",
    "op_setting_1",
    "op_setting_2",
    "op_setting_3"
] + sensor_columns

target_column = "RUL"

split by engine, not random rows

In [0]:
# Split data by engine/unit to avoid data leakage
import random
from sklearn.model_selection import train_test_split

engine_ids = [row["unit_number"] for row in train_df.select("unit_number").distinct().collect()]

random.seed(42)
random.shuffle(engine_ids)

split_index = int(len(engine_ids) * 0.8)

train_engine_ids = engine_ids[:split_index]
validation_engine_ids = engine_ids[split_index:]

train_split_df = train_df.filter(F.col("unit_number").isin(train_engine_ids))
validation_df = train_df.filter(F.col("unit_number").isin(validation_engine_ids))

print("Training engines:", len(train_engine_ids))
print("Validation engines:", len(validation_engine_ids))

print("Training rows:", train_split_df.count())
print("Validation rows:", validation_df.count())

In [0]:
from sklearn.model_selection import train_test_split
train_test_split(train_df)

In [0]:
engine_ids = [
    row["unit_number"]
    for row in train_df
    .select("unit_number")
    .distinct()
    .orderBy("unit_number")
    .collect()
]

split_index = int(len(engine_ids) * 0.8)

train_engines = engine_ids[:split_index]
val_engines = engine_ids[split_index:]

ml_train = train_df.filter(
    F.col("unit_number").isin(train_engines)
)

ml_val = train_df.filter(
    F.col("unit_number").isin(val_engines)
)

print("Training engines:", len(train_engines))
print("Validation engines:", len(val_engines))

Building the ML Model

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

In [0]:
assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

train_vector = assembler.transform(ml_train)
val_vector = assembler.transform(ml_val)

In [0]:
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="RUL",
    numTrees=100,
    maxDepth=10,
    seed=42
)

rf_model = rf.fit(train_vector)

EVALUATION

In [0]:
predictions = rf_model.transform(val_vector)

display(
    predictions.select(
        "unit_number",
        "cycle",
        "RUL",
        "prediction"
    )
)

RMSE

In [0]:
predictions = rf_model.transform(val_vector)

display(
    predictions.select(
        "unit_number",
        "cycle",
        "RUL",
        "prediction"
    )
)

MAE

In [0]:
mae_evaluator = RegressionEvaluator(
    labelCol="RUL",
    predictionCol="prediction",
    metricName="mae"
)

mae = mae_evaluator.evaluate(predictions)

print("MAE:", mae)

R^2

In [0]:
r2_evaluator = RegressionEvaluator(
    labelCol="RUL",
    predictionCol="prediction",
    metricName="r2"
)

r2 = r2_evaluator.evaluate(predictions)

print("R²:", r2)

Linear regression

In [0]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(
    featuresCol="features",
    labelCol="RUL"
)

lr_model = lr.fit(train_vector)

lr_predictions = lr_model.transform(val_vector)

In [0]:
lr_rmse = rmse_evaluator.evaluate(lr_predictions)
lr_mae = mae_evaluator.evaluate(lr_predictions)
lr_r2 = r2_evaluator.evaluate(lr_predictions)

print("Linear Regression")
print("RMSE:", lr_rmse)
print("MAE:", lr_mae)
print("R²:", lr_r2)

RANDOM FOREST

In [0]:
from pyspark.ml.regression import GBTRegressor

gbt = GBTRegressor(
    featuresCol="features",
    labelCol="RUL",
    maxIter=50,
    maxDepth=5,
    seed=42
)

gbt_model = gbt.fit(train_vector)

gbt_predictions = gbt_model.transform(val_vector)

In [0]:
gbt_rmse = rmse_evaluator.evaluate(gbt_predictions)
gbt_mae = mae_evaluator.evaluate(gbt_predictions)
gbt_r2 = r2_evaluator.evaluate(gbt_predictions)

print("Gradient Boosting")
print("RMSE:", gbt_rmse)
print("MAE:", gbt_mae)
print("R²:", gbt_r2)

Create a model comparison table

In [0]:
model_results = spark.createDataFrame(
    [
        ("Linear Regression", lr_rmse, lr_mae, lr_r2),
        ("Random Forest", rmse, mae, r2),
        ("Gradient Boosting", gbt_rmse, gbt_mae, gbt_r2)
    ],
    ["model", "RMSE", "MAE", "R2"]
)

display(model_results)

Feature importance

In [0]:
feature_importance = rf_model.featureImportances

importance_data = [
    (feature_columns[i], float(feature_importance[i]))
    for i in range(len(feature_columns))
]

importance_df = spark.createDataFrame(
    importance_data,
    ["feature", "importance"]
)

display(
    importance_df
    .orderBy(F.col("importance").desc())
)

Create final predictions

In [0]:
final_predictions = rf_model.transform(
    assembler.transform(train_df)
)

In [0]:
model_results = spark.table(
    "workspace.default.ml_model_results"
)

feature_importance = spark.table(
    "workspace.default.ml_feature_importance"
)

predictions = spark.table(
    "workspace.default.ml_test_predictions"
)